# Paper 3 — 05 · Capability evaluation (EN + RO)

Measures the alignment tax (METHOD_DESIGN §5.2):

- **EN suite** via `lm-eval-harness`: MMLU (5-shot), ARC-Easy/Challenge,
  HellaSwag, TruthfulQA-MC2.
- **RO suite**: perplexity on a CulturaX-RO held-out shard, Flores-200 RO→EN
  chrF, plus a small RO-QA accuracy probe (~50 facts; reused from Paper 2's
  hallucination split, judged by `gpt-5-mini`).

In [ ]:
%%capture
# Pinned versions match METHOD_DESIGN §4 + requirements.txt. Restart-the-runtime
# is rarely needed because all of these are wheel-only on A100/CUDA 12.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'torchao>=0.16' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    huggingface_hub ipywidgets pyyaml -q


In [ ]:
%%capture
!pip install -U "lm-eval>=0.4.5" sacrebleu sacremoses -q

In [ ]:
import os, json, gc, time, hashlib, math, sys
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PROBE_DIR    = DRIVE_ROOT / "data" / "probes"
PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
RESULTS_DIR  = DRIVE_ROOT / "results"
LOGS_DIR     = DRIVE_ROOT / "logs"
FIG_DIR      = DRIVE_ROOT / "figures"
for d in [PROBE_DIR, PREFS_DIR, SPLITS_DIR, ADAPTERS_DIR, RESULTS_DIR, LOGS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Paper 2 reuse: judges, llm_judge, holdout splits ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- A100 sanity + perf toggles ---
assert torch.cuda.is_available(), "GPU not available -- switch runtime to A100 GPU."
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")
if "A100" not in DEVICE_NAME:
    print(f"WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE/grad-accum below.")

torch.set_float32_matmul_precision("high")          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True               # autotune for fixed shapes

# --- HF / OpenRouter auth (set in Colab Secrets, not in code) ---
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
except Exception as _e:
    print(f"(secrets not configured: {_e}; set HF_TOKEN / OPENROUTER_API_KEY in Colab secrets)")


def load_kwargs_for(family: str) -> dict:
    """A100-tuned dtype + attention impl per model family.

    Why bf16 everywhere: A100 has bf16 tensor cores at the same throughput as
    fp16, bf16 has the dynamic range of fp32 (no overflow on long sequences),
    and bf16 is the training dtype for all anchor families. Using fp16 here
    silently broke Gemma 3 in Paper 2 (953 empty-string outputs).
    """
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        # Gemma 3 sliding-window attention is brittle with flash-attn-2.
        return {**common, "attn_implementation": "eager"}
    # PyTorch SDPA on A100 + bf16 is fast enough for our workloads.
    # flash-attn-2 was tried but its source-compile step costs ~15 min on
    # every cold Colab runtime, with negligible payoff for batch sizes <= 16.
    return {**common, "attn_implementation": "sdpa"}


def family_of(anchor_id: str) -> str:
    a = anchor_id.lower()
    if "qwen2.5" in a: return "qwen2.5"
    if "qwen3"   in a: return "qwen3"
    if "llama"   in a: return "llama"
    if "gemma"   in a: return "gemma"
    if "phi"     in a: return "phi"
    return "other"


def short_of(anchor_id: str) -> str:
    return (anchor_id.split("/")[-1].lower()
            .replace("-instruct", "").replace("-it", ""))

print("Bootstrap done.")


## Configuration

In [ ]:
ANCHOR    = "Qwen/Qwen2.5-3B-Instruct"
CONDITION = "rd-dpo-k4"
SEED      = 17
EN_TASKS  = ["mmlu", "arc_easy", "arc_challenge", "hellaswag", "truthfulqa_mc2"]
EN_BS     = 8

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

RUN_TAG = f"{short}__{CONDITION}__seed{SEED}"
ADAPTER = ADAPTERS_DIR / RUN_TAG
OUT     = RESULTS_DIR / f"{RUN_TAG}__capability.json"

## Build a merged-weights snapshot for lm-eval

`lm-eval-harness` consumes a HF model id or a local path. For LoRA conditions we
merge the adapter into the base weights and write a transient snapshot under
`/content/merged/<run_tag>/`. For `base` and `dpo-full` we point lm-eval at the
original location directly.

In [ ]:
MERGED = Path("/content/merged") / RUN_TAG
MERGED.mkdir(parents=True, exist_ok=True)

if CONDITION == "base":
    model_path_for_eval = ANCHOR
elif CONDITION == "dpo-full":
    model_path_for_eval = str(ADAPTER)
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    if not (MERGED / "config.json").exists():
        tok = AutoTokenizer.from_pretrained(ANCHOR)
        base = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
        merged = PeftModel.from_pretrained(base, str(ADAPTER)).merge_and_unload()
        merged.save_pretrained(MERGED)
        tok.save_pretrained(MERGED)
        del merged, base; gc.collect(); torch.cuda.empty_cache()
    model_path_for_eval = str(MERGED)
print(f"lm-eval target: {model_path_for_eval}")

## EN suite via lm-eval-harness

In [ ]:
import subprocess

en_out = RESULTS_DIR / f"{RUN_TAG}__lm_eval_en"
en_out.mkdir(parents=True, exist_ok=True)

cmd = [
    "lm_eval",
    "--model", "hf",
    "--model_args", f"pretrained={model_path_for_eval},dtype=bfloat16,trust_remote_code=False",
    "--tasks", ",".join(EN_TASKS),
    "--batch_size", str(EN_BS),
    "--output_path", str(en_out),
    "--seed", str(SEED),
]
print(" ".join(cmd))
rc = subprocess.run(cmd).returncode
if rc != 0: raise RuntimeError(f"lm-eval failed with exit code {rc}")

en_json = next(en_out.rglob("results*.json"))
en_results = json.loads(en_json.read_text())
en_summary = {t: en_results["results"].get(t, {}) for t in EN_TASKS}
print(json.dumps(en_summary, indent=2))

## RO suite — perplexity, Flores RO→EN, RO-QA accuracy

In [ ]:
from datasets import load_dataset
import sacrebleu, math
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(model_path_for_eval, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token
mdl = AutoModelForCausalLM.from_pretrained(model_path_for_eval, **load_kwargs_for(family))
mdl.eval()

print("RO perplexity (CulturaX-RO sample, 100 docs)...")
ds = load_dataset("uonlp/CulturaX", "ro", split="train", streaming=True)
docs = []
for i, ex in enumerate(ds):
    if i >= 100: break
    docs.append(ex["text"][:2000])

nll, ntok = 0.0, 0
@torch.inference_mode()
def doc_nll(text):
    ids = tok(text, return_tensors="pt", truncation=True, max_length=1024).to(mdl.device)
    labels = ids["input_ids"].clone()
    out = mdl(**ids, labels=labels)
    return out.loss.item() * labels.numel(), labels.numel()
for d in docs:
    l, n = doc_nll(d); nll += l; ntok += n
ro_ppl = math.exp(nll / max(ntok, 1))
print(f"  perplexity = {ro_ppl:.2f} (n_tokens={ntok})")

print("Flores-200 RO->EN translation (devtest, 100 sentences)...")
flores = load_dataset("facebook/flores", "ron_Latn-eng_Latn", split="devtest")
src = [ex["sentence_ron_Latn"] for ex in flores][:100]
ref = [ex["sentence_eng_Latn"] for ex in flores][:100]

@torch.inference_mode()
def translate(texts, batch=8):
    outs = []
    for i in range(0, len(texts), batch):
        batch_in = texts[i:i + batch]
        prompts = [
            tok.apply_chat_template(
                [{"role": "user",
                  "content": f"Translate the following Romanian sentence to English. Return only the translation, no commentary.\n\n{t}"}],
                tokenize=False, add_generation_prompt=True,
            ) for t in batch_in
        ]
        ids = tok(prompts, return_tensors="pt", padding=True,
                  truncation=True, max_length=1024).to(mdl.device)
        out = mdl.generate(**ids, max_new_tokens=128, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        input_len = ids["input_ids"].shape[1]
        for seq in out:
            outs.append(tok.decode(seq[input_len:], skip_special_tokens=True).strip())
    return outs
hyp  = translate(src)
chrf = sacrebleu.corpus_chrf(hyp, [ref]).score
print(f"  chrF = {chrf:.2f}")

print("RO-QA probe (50 facts from Paper 2 hallucination split)...")
hall = [json.loads(l) for l in open(PAPER2_ROOT / "benchmark" / "expanded" / "hallucination.jsonl",
                                    encoding="utf-8") if json.loads(l).get("ground_truth")][:50]
qa_correct = 0
from llm_judge import Judge
qa_judge = Judge(model="openai/gpt-5-mini", cache_namespace="paper3_ro_qa")
QA_SYS = ('You are scoring whether a model answer matches a Romanian factual '
          'reference. Reply with strict JSON {"label": "correct|incorrect", '
          '"justification": "..."}.')
for it in hall:
    prompt = tok.apply_chat_template(
        [{"role": "user", "content": it["text_ro"]}],
        tokenize=False, add_generation_prompt=True,
    )
    ids = tok(prompt, return_tensors="pt").to(mdl.device)
    out = mdl.generate(**ids, max_new_tokens=128, do_sample=False,
                       pad_token_id=tok.pad_token_id)
    ans = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    v = qa_judge.classify(
        system=QA_SYS,
        user=f"QUESTION: {it['text_ro']}\nREFERENCE: {it['ground_truth']}\nANSWER: {ans}",
    )
    if v.get("label") == "correct": qa_correct += 1
ro_qa_acc = qa_correct / len(hall)
print(f"  RO-QA accuracy = {ro_qa_acc:.3f}")

OUT.write_text(json.dumps({
    "anchor_model": ANCHOR, "condition": CONDITION, "seed": SEED,
    "capability_en": en_summary,
    "capability_ro": {
        "culturax_ppl": round(ro_ppl, 3),
        "flores_ro_en_chrf": round(chrf, 3),
        "ro_qa_accuracy": round(ro_qa_acc, 3),
    },
    "evaluated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}, indent=2))
print(f"wrote {OUT}")